In [1]:
from pynq.overlays.base import BaseOverlay

base = BaseOverlay("base.bit")

In [56]:
import time, multiprocessing, os, logging

from pynq.lib.pmod.pmod_pwm import Pmod_PWM
from pynq.lib.pmod.pmod_io import Pmod_IO

class RC_Car:
    def __init__(self, log_level = logging.INFO):
        self.logger = logging.getLogger("PYNQ-Tag")
        self.logger.setLevel(log_level)
        file_handler = logging.FileHandler("PYNQ-Tag.log", mode="w")

        file_handler.setFormatter(
            logging.Formatter(
                "%(asctime)s.%(msecs)03d - %(levelname)s - %(message)s",
                datefmt="%Y-%m-%d %H:%M:%S"))
        self.logger.addHandler(file_handler)

        self.laser = self.Laser(logger = self.logger)
        self.ir_receiver = self.IR_Receiver(logger = self.logger)

    class Laser:
        def __init__(self, logger, signal_pin = 3):
            # IR transmitter "laser" is on PMODB and connected to
            # pin 3
            self.pwm = Pmod_PWM(base.PMODB, signal_pin)

            self.pwm_period_usec = 26 # 26 usec is about 38.4 KHz
            self.pwm_duty_cycle = 50
            self.logger = logger

            self.logger.info(f"Initialized Laser class")
            self.logger.debug(f"Laser signal pin: {signal_pin}")
            self.logger.debug(
                f"PWM frequency: {1000.0 / self.pwm_period_usec:.2f} KHz")
            self.logger.debug(f"PWM duty cycle: {self.pwm_duty_cycle}%")

        def shoot(self, pulse_time_msec = 250):
            self.logger.info(f"Laser shot!")
            self.logger.debug(
                f"Laser pulse duration: {pulse_time_msec / 1000} seconds")

            # Shoot a quarter second pulse "laser"
            self.pwm.generate(self.pwm_period_usec, self.pwm_duty_cycle)
            time.sleep(pulse_time_msec / 1000)
            self.pwm.stop()

    class IR_Receiver():
        def __init__(self, logger, signal_pin = 3):
            # IR receiver is on PMODA and connected to pin 3
            self.io = Pmod_IO(base.PMODA, signal_pin, "in")
            self.logger = logger

            # Start a process which indicates a hit
            self.proc = multiprocessing.Process(
                            target=self.notify_hit_p, args=())
            self.proc.start()
            os.system("taskset -p {} {}".format(0x2, self.proc.pid))

            self.logger.info(f"Initialized IR Receiver class")
            self.logger.debug(f"IR receiver signal pin: {signal_pin}")

        def notify_hit_p(self):
            prev_read = self.io.read()
            while True:
                curr_read = self.io.read()
                if curr_read == 0 and curr_read != prev_read:
                    # TODO: This will send a kill signal
                    self.logger.info("Hit!")
                    print("Hit!")
                prev_pread = curr_read
                time.sleep(0.2)

car = RC_Car(logging.DEBUG)

pid 2202's current affinity mask: 3
pid 2202's new affinity mask: 2
Hit!
Hit!
Hit!
Hit!
Hit!


In [61]:
car.laser.shoot()